In [1]:
import requests, os
from astropy.time import Time

from ampel.lsst.alert.load import LSSTArchiveAlertLoader

In [2]:
token = "0bdfc911-21a7-42dd-ae90-65bfcf1f1b99"
header = {"Authorization": "bearer "+token}

In [3]:
endpoint = 'https://ampel-dev.ia.zeuthen.desy.de/api/lsst/archive/v1/streams/from_query'

In [4]:
# import json
# path_json = '/Users/cristozilleruelo/Ampel-LSST-archive-hpx/tests/test-data/quary_dict.json'
# with open(path_json, 'r') as file:
#     hpx_query = json.load(file)
# hpx_query

In [7]:
# ra = 75.89
# dec = 14.63
# date_str = '2026-02-15'
# query = {
# "cone": {
#     "ra": ra,
#     "dec": dec,
#     "radius": 1. / 120
#   },
# "candidate": {
#     # "drb": {
#     #   "$gt": 0.995
#     # },
#     "ndethist": {
#       "$gt": 5,
#       "$lte": 10
#     },
#     'jd': {
#         "$gt": Time(
#                     date_str,
#                     format='iso',
#                     scale="utc",
#                 ).jd
#     }
# },
#     "condition": {'''diaObject.nDiaSources >= 10
#               and diaSource.snr >= 100
#               and diaSource.reliability >= 0.97
#               and diaSource.midpointMjdTai > 61030'''}
#   "isdiffpos": {"$in": ["t", "1"]},
# }

# query = {
#     "condition": "diaObject.nDiaSources >= 10 and diaSource.snr >= 100 and diaSource.reliability >= 0.97 and diaSource.midpointMjdTai > 61030"
# }
query = {
  "exclude": [
    "cutoutScience",
    "cutoutTemplate",
    "cutoutDifference"
  ],
  "condition": "diaSource.reliability > 0.999 and diaSource.visit >= 2026030100000",
  "location": {
    "ra": 149.9287,
    "dec": 0.4177,
    "radius": 10. /3600 #10 arcsec
  },
  "limit": 1
}

In [9]:


response = requests.post(endpoint, headers=header, json=query, verify=False)#, max_retries=retries)

ConnectTimeout: HTTPSConnectionPool(host='ampel-dev.ia.zeuthen.desy.de', port=443): Max retries exceeded with url: /api/lsst/archive/v1/streams/from_query (Caused by ConnectTimeoutError(<HTTPSConnection(host='ampel-dev.ia.zeuthen.desy.de', port=443) at 0x10ffc1ee0>, 'Connection to ampel-dev.ia.zeuthen.desy.de timed out. (connect timeout=None)'))

In [ ]:
if not response.ok:
    print( 'Query creation failed.')

In [ ]:
# The full response contains the resume token as well as the chunk size, i.e.
# how many alerts will be return in each call to the alert iterator.
response.json()

In [ ]:
resume_token = response.json()['resume_token']

In [ ]:
# The loader config contains the resume_token as stream identifier
config = {'archive':"https://ampel-dev.ia.zeuthen.desy.de/api/lsst/archive/v1/", 
          "stream":resume_token}

In [ ]:
try:
    alertlist = []
    alertloader = LSSTArchiveAlertLoader(**config)
    alerts = alertloader.get_alerts()
    for alert in alerts:
        alertlist.append(alert)
        print(alert['objectId'], alert['candidate']['jd'])
except requests.exceptions.HTTPError as e:
    status_code = e.response.status_code
    if status_code==423:
        print('HTTP error {}: likely caused by server staging process. Wait and try again.'.format(status_code) )
    else:
        raise e